In [ ]:
import torch
import torch.nn as nn
import math
import h5py
from torch.utils.data import Dataset, DataLoader

In [ ]:
path = r"your_path_dataset"

In [ ]:
path_train = path + "/traintest_hcd.hdf5"
path_test  = path + "/holdout_hcd.hdf5"

In [ ]:
import h5py

In [ ]:
f = h5py.File(path_train, "r")
for key in list(f.keys()):
    print(key)


In [ ]:
intensity = f["intensities_raw"][0]
intensity.reshape(29,6)

In [ ]:
print(intensity.max())
print(intensity.min())


In [ ]:
sequence = f["sequence_integer"][0]
sequence

In [ ]:
masses_raw = f["masses_raw"][0]
len(masses_raw)



In [ ]:
# import h5py
# import torch
# from tqdm import tqdm

# hdf5_path = path_train
# save_path = "/kaggle/working/targets.pt"

# chunk_size = 50000  # 50k mẫu/lần

# with h5py.File(hdf5_path, "r") as data:

#     N = len(data["sequence_integer"])
#     targets = torch.zeros(N, 29, 6, dtype=torch.float32)

#     for start in tqdm(range(0, N, chunk_size)):

#         end = min(start + chunk_size, N)

#         intensities = torch.tensor(
#             data["intensities_raw"][start:end],
#             dtype=torch.float32
#         )

#         seqs = torch.tensor(
#             data["sequence_integer"][start:end]
#         )

#         lengths = (seqs != 0).sum(dim=1)

#         frag_range = torch.arange(1, 30).unsqueeze(0).repeat(end-start, 1)

#         valid_mask = frag_range < lengths.unsqueeze(1)

#         idx_base_b = frag_range - 1
#         idx_b = torch.stack([
#             0 * 29 + idx_base_b,
#             1 * 29 + idx_base_b,
#             2 * 29 + idx_base_b
#         ], dim=2).clamp(min=0)

#         y_pos = lengths.unsqueeze(1) - frag_range
#         idx_base_y = y_pos - 1

#         idx_y = torch.stack([
#             87 + 0 * 29 + idx_base_y,
#             87 + 1 * 29 + idx_base_y,
#             87 + 2 * 29 + idx_base_y
#         ], dim=2).clamp(min=0)

#         expanded = intensities.unsqueeze(1).expand(-1, 29, -1)

#         b_vals = torch.gather(expanded, 2, idx_b)
#         y_vals = torch.gather(expanded, 2, idx_y)

#         chunk_target = torch.zeros(end-start, 29, 6)
#         chunk_target[:, :, :3] = b_vals
#         chunk_target[:, :, 3:] = y_vals
#         chunk_target[~valid_mask] = 0

#         targets[start:end] = chunk_target

# torch.save(targets, save_path)

In [ ]:
import h5py
import torch
from tqdm import tqdm

hdf5_path = path_train
save_path = "/kaggle/working/targets.pt"

chunk_size = 50000

with h5py.File(hdf5_path, "r") as data:

    N = len(data["sequence_integer"])
    targets = torch.zeros(N, 29, 6, dtype=torch.float32)

    for start in tqdm(range(0, N, chunk_size)):

        end = min(start + chunk_size, N)

        intensities = torch.tensor(
            data["intensities_raw"][start:end],
            dtype=torch.float32
        )

        seqs = torch.tensor(
            data["sequence_integer"][start:end]
        )

        lengths = (seqs != 0).sum(dim=1)

        frag_range = torch.arange(1, 30).unsqueeze(0).repeat(end-start, 1)

        valid_mask = frag_range < lengths.unsqueeze(1)

        # reshape 174 → (29,6)
        chunk_target = intensities.view(end-start, 29, 6)

        chunk_target[chunk_target < 0] = 0

        # mask fragment không hợp lệ
        chunk_target[~valid_mask] = 0

        targets[start:end] = chunk_target

torch.save(targets, save_path)

In [ ]:
# import torch

# def prosit_to_pdeep_fast(intensity_174, seq_len, max_frag=29):
#     """
#     intensity_174: (174,)
#     seq_len: số amino acid thật (không tính padding)
#     return: (29, 6)  -- đã pad sẵn
#     """

#     intensity = torch.as_tensor(intensity_174, dtype=torch.float32)

#     # tạo target cố định 29 cleavage
#     target = torch.zeros(29, 6, dtype=torch.float32)

#     if seq_len <= 1:
#         return target

#     L = seq_len
#     frag_range = torch.arange(1, L)          # 1 .. L-1
#     L_minus_1 = L - 1

#     # ---------- b ions ----------
#     valid_b = frag_range <= max_frag
#     idx_base_b = frag_range - 1

#     idx_b = torch.stack([
#         0 * 29 + idx_base_b,
#         1 * 29 + idx_base_b,
#         2 * 29 + idx_base_b
#     ], dim=1)  # shape (L-1, 3)

#     target[:L_minus_1, :3][valid_b] = intensity[idx_b[valid_b]]

#     # ---------- y ions ----------
#     y_pos = L - frag_range
#     valid_y = y_pos <= max_frag
#     idx_base_y = y_pos - 1

#     idx_y = torch.stack([
#         87 + 0 * 29 + idx_base_y,
#         87 + 1 * 29 + idx_base_y,
#         87 + 2 * 29 + idx_base_y
#     ], dim=1)  # shape (L-1, 3)

#     target[:L_minus_1, 3:][valid_y] = intensity[idx_y[valid_y]]

#     return target

In [ ]:
import torch

def prosit_to_pdeep_fast(intensity_174, seq_len, max_frag=29):
    """
    intensity_174: (174,)
    seq_len: số amino acid thật
    return: (29,6)
    """

    intensity = torch.as_tensor(intensity_174, dtype=torch.float32)

    # reshape trực tiếp
    target = intensity.view(29, 6).clone()

    if seq_len <= 1:
        return torch.zeros(29,6)

    L = seq_len
    valid_len = min(L - 1, max_frag)
    
    target[target < 0] = 0
    
    # mask fragment không tồn tại
    target[valid_len:] = 0

    return target

In [ ]:
import h5py
import torch
from tqdm import tqdm

hdf5_path = path_train

with h5py.File(hdf5_path, "r") as data:

    print("Loading sequences...")
    all_seqs = torch.tensor(
        data["sequence_integer"][:],
        dtype=torch.long
    )

    print("Loading charge...")
    charge_onehot = torch.tensor(
        data["precursor_charge_onehot"][:]
    )
    all_charge = (
        charge_onehot.argmax(dim=1) * 0.1 + 0.1
    ).unsqueeze(1).float()

    print("Loading NCE...")
    all_nce = torch.tensor(
        data["collision_energy_aligned_normed"][:],
        dtype=torch.float32
    )

print("Saving...")
torch.save(all_seqs, "/kaggle/working/seq.pt")
torch.save(all_charge, "/kaggle/working/charge.pt")
torch.save(all_nce, "/kaggle/working/nce.pt")

print("Done.")

In [ ]:
import h5py
import torch
from torch.utils.data import Dataset

class PDeepDataset(Dataset):
    def __init__(self):
        self.seqs = torch.load("/kaggle/working/seq.pt")
        self.charge = torch.load("/kaggle/working/charge.pt")
        self.nce = torch.load("/kaggle/working/nce.pt")
        self.targets = torch.load("/kaggle/working/targets.pt")

    def __len__(self):
        return self.targets.shape[0]

    def __getitem__(self, idx):

        seq = self.seqs[idx]
        charge = self.charge[idx]
        nce = self.nce[idx]
        target = self.targets[idx]
        mask = (seq != 0)[:-1]
        last = mask.sum() - 1
        mask[last] = False
        mask = mask.float()

        return seq, charge, nce, target, mask

In [ ]:
dataset = PDeepDataset()
len(dataset)

In [ ]:
seq, charge, nce, target , mask = dataset[1]

print("Sequence shape:", seq.shape)
print("Charge:", charge)
print("NCE:", nce)
print("Target shape:", target.shape)

In [ ]:
seq[:]

In [ ]:
mask[:]


In [ ]:
loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [ ]:
seq, charge, nce, target, mask = next(iter(loader))

print("seq shape:", seq.shape)
print("charge shape:", charge.shape)
print("nce shape:", nce.shape)
print("target shape:", target.shape)
print("mask shape:", mask.shape)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(max_len) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
class MetaEmbedding(nn.Module):
    def __init__(self, meta_dim=8):
        super().__init__()
        self.linear = nn.Linear(1, meta_dim - 1)

    def forward(self, charge, nce):
        meta = self.linear(nce)
        meta = torch.cat([meta, charge], dim=1)
        return meta

In [ ]:
class ModelMS2Transformer(nn.Module):
    def __init__(
        self,
        aa_vocab_size,
        mod_dim,
        num_frag_types,
        num_modloss_types=0,
        hidden=256,
        dropout=0.1,
        nlayers=4,
        mask_modloss=True
    ):
        super().__init__()

        self.num_modloss = num_modloss_types
        self.num_non_modloss = num_frag_types - num_modloss_types
        self.mask_modloss = mask_modloss

        meta_dim = 8

        # ==== Input embedding ====
        self.aa_embedding = nn.Embedding(
            aa_vocab_size,
            hidden - meta_dim - mod_dim
        )

        if mod_dim > 0:
            self.mod_linear = nn.Linear(mod_dim, mod_dim)
        else:
            self.mod_linear = None

        self.pos_encoder = PositionalEncoding(hidden - meta_dim)

        self.meta_embedding = MetaEmbedding(
            meta_dim=meta_dim
        )

        # ==== Transformer backbone ====
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=8,
            dim_feedforward=hidden * 4,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=nlayers
        )

        self.dropout = nn.Dropout(dropout)

        # ==== Main head ====
        self.output_head = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.PReLU(),
            nn.Linear(64, self.num_non_modloss)
        )

        # ==== Modloss branch ====
        if num_modloss_types > 0:
            modloss_layer = nn.TransformerEncoderLayer(
                d_model=hidden,
                nhead=8,
                dim_feedforward=hidden * 4,
                dropout=dropout,
                batch_first=True
            )

            self.modloss_transformer = nn.TransformerEncoder(
                modloss_layer,
                num_layers=1
            )

            self.modloss_head = nn.Sequential(
                nn.Linear(hidden, 64),
                nn.PReLU(),
                nn.Linear(64, num_modloss_types)
            )
        else:
            self.modloss_transformer = None
            self.modloss_head = None
    def forward(self, aa_idx, mod_x, charge, nce):
        B, L = aa_idx.shape

        # ==== AA + Mod embedding ====
        aa_emb = self.aa_embedding(aa_idx)
        if self.mod_linear is not None:
            mod_emb = self.mod_linear(mod_x)
            seq_x = torch.cat([aa_emb, mod_emb], dim=2)
        else:
            seq_x = aa_emb

        seq_x = self.pos_encoder(seq_x)

        # ==== Meta embedding ====
        meta = self.meta_embedding(charge, nce)
        meta = meta.unsqueeze(1).repeat(1, L, 1)

        x = torch.cat([seq_x, meta], dim=2)

        # ==== Backbone ====
        hidden = self.transformer(x)

        # residual trick (paper exact)
        hidden = self.dropout(hidden + 0.2 * x)

        # ==== Main head ====
        out_main = self.output_head(hidden)

        # ==== Modloss ====
        if self.num_modloss > 0:
            if self.mask_modloss:
                zeros = torch.zeros(
                    B, L, self.num_modloss,
                    device=x.device
                )
                out = torch.cat([out_main, zeros], dim=2)
            else:
                modloss_x = self.modloss_transformer(x)
                modloss_x = modloss_x + hidden
                modloss_out = self.modloss_head(modloss_x)
                out = torch.cat([out_main, modloss_out], dim=2)
        else:
            out = out_main

        # remove first cleavage position
        return out[:, 1:, :]
print("hello1")

In [ ]:
model = ModelMS2Transformer(
    aa_vocab_size=30,   # kiểm tra vocab size thật nếu cần
    mod_dim=0,
    num_frag_types=6,
    num_modloss_types=0,
    hidden=256
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

seq, charge, nce, target, mask = next(iter(loader))

seq = seq.to(device)
charge = charge.to(device)
nce = nce.to(device)

pred = model(seq, None, charge, nce)

print("pred shape:", pred.shape)
print("target shape:", target.shape)

In [ ]:
print(pred[0])
print(target[0])

In [ ]:
print(mask[0])

In [ ]:
mask = mask.unsqueeze(-1)
print(mask.shape)
print(mask[0])

In [ ]:
mask = mask.cuda()
target = target.cuda()

In [ ]:
print(pred.device)
print(target.device)
print(mask.device)

In [ ]:
loss = torch.abs(pred - target) * mask
loss[0]

In [ ]:
loss.shape

In [ ]:
loss_per_sample = loss.sum(dim=(1,2))
loss_per_sample 

In [ ]:
valid = mask.sum(dim=(1,2))
valid

In [ ]:
valid.clamp(min=1)


In [ ]:
loss_per_sample = loss_per_sample / valid.clamp(min=1)
loss_per_sample

In [ ]:
def masked_l1_loss(pred, target, mask):

    mask = mask.unsqueeze(-1)

    loss = torch.abs(pred - target) * mask

    # sum fragment + ion
    loss_per_sample = loss.sum(dim=(1,2))

    # số phần tử hợp lệ mỗi peptide
    valid = mask.sum(dim=(1,2))

    loss_per_sample = loss_per_sample / valid.clamp(min=1)

    return loss_per_sample.mean()

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

seq, charge, nce, target, mask = next(iter(loader))

seq = seq.to(device)
charge = charge.to(device)
nce = nce.to(device)
target = target.to(device)
mask = mask.to(device)
# Chuyển tất cả -1 trong target về 0 trước khi tính Loss


pred = model(seq, None, charge, nce)

loss = masked_l1_loss(pred, target, mask)
print("initial loss:", loss.item())
print("min:", pred.min().item())
print("max:", pred.max().item())
print("target min:", target.min().item())
print("target max:", target.max().item())

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


for step in range(200):
    pred = model(seq, None, charge, nce)
    loss = masked_l1_loss(pred, target, mask)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        print(step, loss.item())

In [ ]:
print(f"Số lượng giá trị > 0 trong Pred: {(pred > 0).sum().item()}")

In [ ]:
print(pred[0][0])
print(target[0][0])

In [ ]:
from torch.utils.data import random_split

train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [ ]:
train_loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True,          # tạm tắt shuffle khi dùng HDF5
    num_workers=4,
    pin_memory=True,
    
)

val_loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True,          # tạm tắt shuffle khi dùng HDF5
    num_workers=4,
    pin_memory=True,
    
)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(next(model.parameters()).device)

In [ ]:
import time
start = time.time()

for i, _ in enumerate(train_loader):
    if i == 3:
        break

print("3 batch load time:", time.time() - start)

In [ ]:
best_val = float("inf")

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,      # giảm lr ×0.5
    patience=2,      # 2 epoch không cải thiện
    min_lr=1e-6,
)

In [ ]:
for epoch in range(8):

    # ---- TRAIN ----
    model.train()
    train_loss = 0

    for seq, charge, nce, target, mask in train_loader:

        seq = seq.to(device)
        charge = charge.to(device)
        nce = nce.to(device)
        target = target.to(device)
        mask = mask.to(device)
       

        pred = model(seq, None, charge, nce)

        loss = masked_l1_loss(pred, target, mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ---- VALIDATION ----
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for seq, charge, nce, target, mask in val_loader:

            seq = seq.to(device)
            charge = charge.to(device)
            nce = nce.to(device)
            target = target.to(device)
            mask = mask.to(device)


            pred = model(seq, None, charge, nce)
           
            loss = masked_l1_loss(pred, target, mask)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    # ---- scheduler ----
    scheduler.step(val_loss)

    # ---- save best model ----
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_ms2_model.pt")
        print("Saved best model")

    lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch} | "
        f"Train {train_loss:.4f} | "
        f"Val {val_loss:.4f} | "
        f"LR {lr:.2e}"
    )

In [ ]:
path_test  = path + "/holdout_hcd.hdf5"
path_test

In [ ]:
f = h5py.File(path_test, "r")
for key in list(f.keys()):
    print(key)


In [ ]:
sequence_test = f["sequence_integer"][:]
intensity_test = f["intensities_raw"][:]
nce_test = f["collision_energy_aligned_normed"][:]
charge_test = f["precursor_charge_onehot"][:]


In [ ]:
intensity_test[0]

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. Chuyển đổi Charge One-hot sang số thực (0.1, 0.2, ...)
charge_tensor = torch.tensor(charge_test, dtype=torch.float32)
all_charge_test = (charge_tensor.argmax(dim=1) * 0.1 + 0.1).unsqueeze(1)

# 2. Chuyển các thành phần khác sang Tensor
all_seq_test = torch.tensor(sequence_test, dtype=torch.long)
all_nce_test = torch.tensor(nce_test, dtype=torch.float32)
all_intensity_test = torch.tensor(intensity_test, dtype=torch.float32)

# 3. Tạo Mask (để biết chiều dài thực của peptide)
all_mask_test = (all_seq_test != 0).float()

# 4. Đóng gói vào DataLoader
test_dataset = TensorDataset(all_seq_test, all_charge_test, all_nce_test, all_intensity_test, all_mask_test)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)



In [ ]:
def calculate_sa(t, p):
    # Spectral Angle đo góc giữa 2 vector phổ
    # Càng gần 1 càng tốt
    norm_t = np.linalg.norm(t)
    norm_p = np.linalg.norm(p)
    if norm_t == 0 or norm_p == 0: return 0
    
    cos_theta = np.dot(t, p) / (norm_t * norm_p)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    return 1 - (2 * np.arccos(cos_theta) / np.pi)

In [ ]:
import numpy as np
from scipy.stats import pearsonr
from tqdm import tqdm

model.eval()
all_pccs = []
all_sas = []

print("Đang bắt đầu Evaluate tập Test...")

with torch.no_grad():
    for seq, charge, nce, raw_int, mask in tqdm(test_loader):
        # Đẩy input lên GPU
        seq_gpu = seq.to(device)
        charge_gpu = charge.to(device)
        nce_gpu = nce.to(device)
        
        # 1. Model dự đoán (Output: Batch, 29, 6)
        with torch.amp.autocast('cuda'):
            preds = model(seq_gpu, None, charge_gpu, nce_gpu)
        
        preds_np = preds.cpu().numpy()
        raw_int_np = raw_int.numpy() 
        mask_np = mask.numpy()
        
        # 2. Xử lý từng mẫu trong batch
        # 2. Xử lý từng mẫu trong batch
        for i in range(preds_np.shape[0]):
            seq_len = int(mask_np[i].sum())
            if seq_len <= 1: continue
            
            # Chuyển đổi sang format pDeep (29, 6)
            target_29_6 = prosit_to_pdeep_fast(raw_int_np[i], seq_len).numpy()

            # --- SỬA Ở ĐÂY ---
            target_29_6[target_29_6 < 0] = 0
           
            p = preds_np[i, :seq_len-1, :].flatten()
            t = target_29_6[:seq_len-1, :].flatten()
            
            p = np.maximum(p, 0) 
            
            # 3. Tính PCC và SA
            if np.std(t) > 1e-12 and np.std(p) > 1e-12:
                pcc, _ = pearsonr(t, p)
                sa = calculate_sa(t, p)
                all_pccs.append(pcc)
                all_sas.append(sa)
            else:
                # Nếu phổ trống không (toàn 0) thì cho Metrics bằng 0 
                # thay vì bỏ qua (continue) để số lượng mẫu khớp 100%
                all_pccs.append(0)
                all_sas.append(0)

print("\n" + "="*30)
print(f"Mean PCC:   {np.mean(all_pccs):.4f}")
print(f"Median PCC: {np.median(all_pccs):.4f}")
print(f"Mean SA: {np.mean(all_sas):.4f}")
print(f"Median SA: {np.median(all_sas):.4f}")


print("="*30)

In [ ]:
test_t = np.array([1.0, 0.5, 0.0])
test_p = np.array([0.9, 0.4, 0.1])
print(calculate_sa(test_t, test_p))
